# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1 
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

In [6]:
import pandas as pd
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())

[9, 10]


Ваши ответы и объяснения:

1. Ответ: 1 1.3333 потому isna().sum() вернет колво пропущенных значений - 1 fillna пропущенное число заменит 0 а mean посчитает среднее значение (1+0+3)/3
2. 2 потому что стрип уберет пробемы в начале и конце и лоуер сделает все маленькими, А и а станут одинаковые = неуникальные
3. думаю все равно 3 (лен возвращает колво строк) тк мы не сохранили результат дроп дубликатов или не сделала инплейс
4. 1005 потому что убрали пробелы, превратили в числа и сделали сумму
5. [9, 10] возвращает месяцы в виде листа



ура (все оказалось правильно)

In [133]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("/Users/gulsatkalbayeva/Documents/dsa-homework/orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("/Users/gulsatkalbayeva/Documents/dsa-homework/orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [134]:
def clean(path): 
    df = pd.read_csv(path)
    print("1. Файл прочли, ее длина:", len(df))
    df['amount'] = pd.to_numeric(df['amount'].str.replace(" ", ""))
    print("2. Amount преобразовали, ее длина:", len(df['amount']))
    print("Внутри аmount вот столько NaN:", df['amount'].isna().sum())
    CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}
    df['city']= df['city'].str.strip().str.lower().map(CITY)
    print("3. City преобразовали, ее длина:", len(df['city']))
    print("Внутри city вот столько NaN:", df['city'].isna().sum())
    df = df.drop_duplicates()
    print("4. Дубли убрали, вот столько строк у нас:", len(df))
    df['order_date'] = pd.to_datetime(df['order_date'])
    print("5. Даты преобразовали в datetime, ее длина:", len(df['order_date']))
    non_test = df['customer_id'] != "TEST"
    df = df[non_test]
    print("6. Убрали ТЕСТ, длина customer_id:", len(df['customer_id']))
    print("7. Order_id уникален:",df['order_id'].is_unique)
    return df


In [135]:
path = "/Users/gulsatkalbayeva/Documents/dsa-homework/orders.csv"
df = clean(path)
len(df)

1. Файл прочли, ее длина: 5463
2. Amount преобразовали, ее длина: 5463
Внутри аmount вот столько NaN: 0
3. City преобразовали, ее длина: 5463
Внутри city вот столько NaN: 0
4. Дубли убрали, вот столько строк у нас: 5283
5. Даты преобразовали в datetime, ее длина: 5283
6. Убрали ТЕСТ, длина customer_id: 5280
7. Order_id уникален: True


5280

## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [136]:
# 2
paid = df['status'] == 'оплачен'
df['order_date'].dt.month.value_counts()
sep = df['order_date'].dt.month == 9
octo = df['order_date'].dt.month == 10
table = pd.Series([df['amount'][sep&paid].sum(),df['amount'][octo&paid].sum()], index= ['sep', 'octo'])
table['change'] = table['octo']/table['sep']-1
print(table)
cities = df['city'].unique()
result = {}
for city in cities:
    mask = (df['city'] == city) & paid
    result[city] = df.loc[mask, 'amount'].sum()
city_t = pd.Series(result)
city_t 
print(city_t)
sep_a = {}
octo_a = {}
for city in cities:
    base = (df['city'] == city) & paid
    sep_a[city] = df['amount'][base&sep].sum()
    octo_a[city] = df['amount'][base&octo].sum()
t = pd.DataFrame({'sep': sep_a, 'octo': octo_a})
t['change'] = t['octo']/t['sep']-1


sep       3.524341e+07
octo      3.441952e+07
change   -2.337714e-02
dtype: float64
Алматы       29144090.0
Астана       17884750.0
Караганда     7020200.0
Шымкент       8602660.0
Атырау        1893850.0
Актобе        5117380.0
dtype: float64


In [137]:
print(t['change'].sort_values())
print("Шымкент падает сильнее всех")

Шымкент     -0.112893
Караганда   -0.075877
Астана      -0.020333
Актобе      -0.014179
Алматы       0.009789
Атырау       0.049643
Name: change, dtype: float64
Шымкент падает сильнее всех


In [138]:
#for myself
df['order_date'].dt.month.value_counts()
len(cities)
print(t['sep'].sum(), table['sep'])
print(t['octo'].sum(), table['octo'])

35243410.0 35243410.0
34419520.0 34419520.0


## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

| Шаг | Строк убрано | Строк осталось | Изменение окт/сент |
|---|---|---|---|
| Грязные данные| 0 | 5 463 | −22,7% |
| Без дублей | 180 | 5 283 | -19.6% |
| Без TEST | 3 | 5 280 | −2,3% |

In [143]:
# 3 
raw = pd.read_csv("/Users/gulsatkalbayeva/Documents/dsa-homework/orders.csv")
raw['amount'] = pd.to_numeric(raw['amount'].str.replace(" ", ""))
raw = raw.drop_duplicates()
raw['order_date'] = pd.to_datetime(raw['order_date'])
sep = raw['order_date'].dt.month == 9
octo = raw['order_date'].dt.month == 10
paid_r = raw['status'] == "оплачен"
len(raw)
sep_raw = raw['amount'][paid_r&sep].sum()
octo_raw = raw['amount'][paid_r&octo].sum()
change_raw = octo_raw/sep_raw - 1
print(f"Изменение без дублей; но с ТЕСТ: {change_raw:+.1%}")
print(f"Изменение без дублей; без ТЕСТ: {table['change']:+.1%}")

Изменение без дублей; но с ТЕСТ: -19.6%
Изменение без дублей; без ТЕСТ: -2.3%


## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [157]:
# 4
print("Сколько пропусков в `delivery_days` по чистым данным?", df['delivery_days'].isna().sum())
df.loc[df['delivery_days'].isna(), ['status', 'city']].value_counts()

Сколько пропусков в `delivery_days` по чистым данным? 158


status   city     
оплачен  Алматы       61
         Астана       35
         Караганда    23
         Шымкент      17
отменён  Астана        5
         Шымкент       5
оплачен  Актобе        4
         Атырау        4
отменён  Алматы        2
         Атырау        1
         Караганда     1
Name: count, dtype: int64

In [179]:
d = df.loc[df['city'] == 'Шымкент', 'delivery_days']
d

5       1.0
19      1.0
25      1.0
30      1.0
31      1.0
       ... 
5409    7.0
5425    7.0
5428    NaN
5441    6.0
5448    5.0
Name: delivery_days, Length: 776, dtype: float64

In [180]:
v1 = d.dropna()
v2 = d.fillna(d.median())
v3 = d.fillna(0)

In [190]:
df['city'].value_counts()

city
Алматы       2116
Астана       1339
Шымкент       776
Караганда     519
Актобе        387
Атырау        143
Name: count, dtype: int64

In [193]:
print(f"(а) `dropna` {(v1<=2).mean():.1%}, (б) `fillna(медиана) {(v2<=2).mean():.1%}`, (в) `fillna(0)` {(v3<=2).mean():.1%}")
print("Какой вариант честный и почему другие вводят в заблуждение?\n")
print("Для меня честный dropna, потому что лучше убрать данные и сделать выводы на том, что у нас есть. \nК тому же; в Шымкенте всего 22 потерянных значений из 776; это нормально убирать часть")
print("fillna(0) прям сильно завышает; доставка за 0 дней это нереально")
print("fillna медианой конечно лучше, чем 0 заполнять, но он начнет даже отмененным заказам ставить медиану")


(а) `dropna` 29.8%, (б) `fillna(медиана) 29.0%`, (в) `fillna(0)` 31.8%
Какой вариант честный и почему другие вводят в заблуждение?

Для меня честный dropna, потому что лучше убрать данные и сделать выводы на том, что у нас есть. 
К тому же; в Шымкенте всего 22 потерянных значений из 776; это нормально убирать часть
fillna(0) прям сильно завышает; доставка за 0 дней это нереально
fillna медианой конечно лучше, чем 0 заполнять, но он начнет даже отмененным заказам ставить медиану


## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [203]:
# 5
CITY2 = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}
df2 = pd.read_csv(path)
df2 = df2.drop_duplicates() #тут не сохранил избавление от дубликатов
df2["city"] = df2["city"].str.lower().str.replace(" ", "").map(CITY2) #тут не убрал пробелы и не избавился от латинской версии

print(df2["city"].nunique(), len(df2))

6 5283


In [204]:
df2["city"].value_counts()

city
Алматы       2119
Астана       1339
Шымкент       776
Караганда     519
Актобе        387
Атырау        143
Name: count, dtype: int64

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [ ]:
dates = pd.to_datetime(df['order_date'].dt.date.unique())   
n_days = dates.day_name().value_counts()                    
days = df['order_date'].dt.day_name().value_counts()
per_day = days / n_days
busy_label = per_day.idxmax()
busy_value = per_day.max()
chill_label = per_day.idxmin()
chill_value = per_day.min()
print("В какой день заказов меньше всего?", chill_label, "| Всего:", chill_value, "заказов в день")
print(f"В какой день заказов больше всего?  {busy_label} | Всего: {busy_value:.2f} заказов в день")
print(f"Разница: в самый занятой день на {busy_value-chill_value:.2f} заказов, {(busy_value/chill_value-1):.1%}  больше")
print("Вывод: разница не такая большая, можно выделять одинаковое количество курьеров в день")
print("Интересно было бы посмотреть по времени")

В какой день заказов меньше всего? Monday | Всего: 85.5 заказов в день
В какой день заказов больше всего?  Thursday | Всего: 87.33 заказов в день
Разница: в самый занятой день на 1.83 заказов, 2.1%  больше
Вывод: разница не такая большая, можно выделять одинаковое количество курьеров в день


In [233]:
days

order_date
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: count, dtype: int64

Я делала проверку с ИИ и он мне сказал, что вычисления нечестные из-за того, что в период нашего датасета одни дни недели встречаются чаще, чем другие и надо считать число среднее число 

In [230]:

days = df['order_date'].dt.day_name().value_counts()
busy_label = days.idxmax()
busy_value = days.max()
chill_label = days.idxmin()
chill_value = days.min()
print("В какой день заказов меньше всего?", chill_label, "| Всего:", chill_value, "заказов")
print("В какой день заказов больше всего?", busy_label, "| Всего:", busy_value, "заказов")
print(f"Разница: в самый занятой день на {busy_value-chill_value} заказов, {(busy_value/chill_value-1):.1%}  больше")


В какой день заказов меньше всего? Monday | Всего: 684 заказов
В какой день заказов больше всего? Thursday | Всего: 786 заказов
Разница: в самый занятой день на 102 заказов, 14.9%  больше


In [ ]:
df['order_date'].dt.hour.value_counts()

order_date
8     379
13    377
21    373
15    372
22    371
17    359
16    356
10    351
9     343
18    343
12    342
20    334
11    331
14    326
19    323
Name: count, dtype: int64

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [ ]:
# 7

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [ ]:
# 8